In [64]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [81]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [82]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [84]:
lr = LinearRegression()

lr.fit(X_train, y_train)

mse_original = mean_squared_error(y_test, lr.predict(X_test))

In [85]:
vif = pd.DataFrame()

vif['Columns'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif

,Columns,VIF
0,MedInc,11.511140
1,HouseAge,7.195917
2,AveRooms,45.993601
3,AveBedrms,43.590314
4,Population,2.935745
5,AveOccup,1.095243
6,Latitude,559.874071
7,Longitude,633.711654


In [86]:
df.corr()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
MedInc,1.000000,-0.119034,0.326895,-0.062040,0.004834,0.018766,-0.079809,-0.015176,0.688075
HouseAge,-0.119034,1.000000,-0.153277,-0.077747,-0.296244,0.013191,0.011173,-0.108197,0.105623
AveRooms,0.326895,-0.153277,1.000000,0.847621,-0.072213,-0.004852,0.106389,-0.027540,0.151948
AveBedrms,-0.062040,-0.077747,0.847621,1.000000,-0.066197,-0.006181,0.069721,0.013344,-0.046701
Population,0.004834,-0.296244,-0.072213,-0.066197,1.000000,0.069863,-0.108785,0.099773,-0.024650
AveOccup,0.018766,0.013191,-0.004852,-0.006181,0.069863,1.000000,0.002366,0.002476,-0.023737
Latitude,-0.079809,0.011173,0.106389,0.069721,-0.108785,0.002366,1.000000,-0.924664,-0.144160
Longitude,-0.015176,-0.108197,-0.027540,0.013344,0.099773,0.002476,-0.924664,1.000000,-0.045967
MedHouseVal,0.688075,0.105623,0.151948,-0.046701,-0.024650,-0.023737,-0.144160,-0.045967,1.000000


---

<h3> 
There are 2 pairs of columns that have high Correlation 

1. AveRooms and AveBedrms
2. Latitude and Longitude 
</h3>

<h4>
Let's talk first about AveRooms and AveBedrms Since they are Heavily Correlated that makes it a redundant Column so we can drop one column or create a new feature of BedroomRatio. Creating new column looks fare approch because it uses both columns data, so no information gets lost  
</h4>

---

In [87]:
df['BedroomRatio'] = df['AveBedrms']/df['AveRooms']
df.drop(columns=['AveBedrms', 'AveRooms'], inplace=True)
df.head()

,MedInc,HouseAge,Population,AveOccup,Latitude,Longitude,MedHouseVal,BedroomRatio
0,8.3252,41.0,322.0,2.555556,37.88,-122.23,4.526,0.146591
1,8.3014,21.0,2401.0,2.109842,37.86,-122.22,3.585,0.155797
2,7.2574,52.0,496.0,2.802260,37.85,-122.24,3.521,0.129516
3,5.6431,52.0,558.0,2.547945,37.85,-122.25,3.413,0.184458
4,3.8462,52.0,565.0,2.181467,37.85,-122.25,3.422,0.172096


---

<h4>Since there is also another Correlated columns Latitude and Longitude but we cannot drop columns because to location of the house matters in the pricing do we create two new columns using the domain knoledge
</h4>

---

In [88]:
df['dist_to_LA'] = np.sqrt((df['Latitude'] - 34.05)**2 + (df['Longitude'] - (-118.24))**2)
df['dist_to_SF'] = np.sqrt((df['Latitude'] - 37.77)**2 + (df['Longitude'] - (-122.42))**2)
df = df.drop(columns=['Latitude', 'Longitude'])
df.head()

,MedInc,HouseAge,Population,AveOccup,MedHouseVal,BedroomRatio,dist_to_LA,dist_to_SF
0,8.3252,41.0,322.0,2.555556,4.526,0.146591,5.530732,0.219545
1,8.3014,21.0,2401.0,2.109842,3.585,0.155797,5.509673,0.219317
2,7.2574,52.0,496.0,2.802260,3.521,0.129516,5.517246,0.196977
3,5.6431,52.0,558.0,2.547945,3.413,0.184458,5.524500,0.187883
4,3.8462,52.0,565.0,2.181467,3.422,0.172096,5.524500,0.187883


---

<h4> Let's check VIF after engineering </h4>

___

In [89]:
vif_after_fix = pd.DataFrame()
vif_after_fix['Columns'] = X.columns
vif_after_fix['VIF_after_fix'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_after_fix

,Columns,VIF_after_fix
0,MedInc,11.511140
1,HouseAge,7.195917
2,AveRooms,45.993601
3,AveBedrms,43.590314
4,Population,2.935745
5,AveOccup,1.095243
6,Latitude,559.874071
7,Longitude,633.711654


In [91]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

In [92]:
X[['BedroomRatio','HouseAge','dist_to_LA','dist_to_SF']].corr()

,BedroomRatio,HouseAge,dist_to_LA,dist_to_SF
BedroomRatio,1.000000,0.136367,-0.133777,0.103103
HouseAge,0.136367,1.000000,-0.026022,-0.105991
dist_to_LA,-0.133777,-0.026022,1.000000,-0.858797
dist_to_SF,0.103103,-0.105991,-0.858797,1.000000


In [93]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [95]:
scaler = StandardScaler()

X_train_fixed = scaler.fit_transform(X_train)
X_test_fixed = scaler.transform(X_test)

print(f"Condition Number Before fixing : {np.linalg.cond(X_train)}")
print(f"Condition Number after fixing : {np.linalg.cond(X_train_fixed)}")

Condition Number Before fixing : 29982.80279605354
Condition Number after fixing : 4.0428104706130545


---

<h4>As you can see Condition Number significantly Dropped.</h4>

now lets apply Linear Regressor to both of the data fixed and unfixed

---

In [96]:
lr2 = LinearRegression()

lr2.fit(X_train_fixed, y_train)

mse_fixed = mean_squared_error(y_test, lr2.predict(X_test_fixed))
print(f"MSE Before Fixing : {mse_original}")
print(f"MSE After Fixing : {mse_fixed}")

MSE Before Fixing : 0.5431489670037237
MSE After Fixing : 0.6033541125071175


---

<h4> 
As you can see that R2 score increased after fixing the multicollinearity
</h4>

---